AI to Draw

In [1]:
import torch
import cv2
import numpy as np
import gradio as gr
from PIL import Image

from diffusers import StableDiffusionControlNetPipeline, ControlNetModel
from transformers import CLIPProcessor, CLIPModel
from transformers import BlipProcessor, BlipForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------
# Caption model (better than fixed labels)
# -------------------------
blip_processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to(device)

# -------------------------
# CLIP embeddings for learning
# -------------------------
clip_model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
).to(device)

clip_processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)

# -------------------------
# ControlNet
# -------------------------
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-scribble",
    torch_dtype=torch.float16
)

pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16
).to(device)

pipe.enable_attention_slicing()

# -------------------------
# Feedback dataset
# -------------------------
embeddings = []
labels = []

# -------------------------
# Caption from sketch
# -------------------------
def caption_image(image):

    inputs = blip_processor(
        image,
        return_tensors="pt"
    ).to(device)

    out = blip_model.generate(**inputs)

    caption = blip_processor.decode(
        out[0],
        skip_special_tokens=True
    )

    return caption


# -------------------------
# Prompt builder
# -------------------------
def build_prompt(caption, style):

    if style == "photorealistic":
        style_prompt = "ultra realistic photo, DSLR, natural lighting, 50mm lens, high detail"

    else:
        style_prompt = style

    return f"{caption}, {style_prompt}"

# -------------------------
# Generate image
# -------------------------
def generate(image, style):

    if isinstance(image, dict):
        image = image["composite"]

    img_np = np.array(image)

    caption = caption_image(image)

    edge = cv2.Canny(img_np, 100, 200)
    edge = np.stack([edge]*3, axis=2)
    edge = Image.fromarray(edge).resize((512,512))

    prompt = build_prompt(caption, style)

    result = pipe(prompt, image=edge).images[0]

    return result, caption


# -------------------------
# Save feedback
# -------------------------
def save_feedback(image, correct_label):

    if isinstance(image, dict):
        image = image["composite"]

    inputs = clip_processor(
        images=image,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        emb = clip_model.get_image_features(**inputs)

    embeddings.append(emb.cpu())
    labels.append(correct_label)

    return f"Saved examples: {len(labels)}"


# -------------------------
# UI
# -------------------------
with gr.Blocks() as demo:

    gr.Markdown("# Sketch → AI Art")

    canvas = gr.ImageEditor(type="pil")

    style = gr.Dropdown([
        "photorealistic",
        "anime",
        "studio ghibli",
        "oil painting",
        "pencil sketch"
    ])

    btn = gr.Button("Generate")

    output = gr.Image()
    caption_box = gr.Textbox(label="AI Description")

    correct_label = gr.Textbox(label="Correction")
    feedback_btn = gr.Button("Save Feedback")
    status = gr.Textbox()

    btn.click(
        generate,
        inputs=[canvas, style],
        outputs=[output, caption_box]
    )

    feedback_btn.click(
        save_feedback,
        inputs=[canvas, correct_label],
        outputs=status
    )

demo.launch()

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1a5b456fae8f57727e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [6]:
import torch
import cv2
import numpy as np
import gradio as gr
from PIL import Image

from diffusers import StableDiffusionControlNetPipeline, ControlNetModel
from transformers import CLIPProcessor, CLIPModel
from transformers import BlipProcessor, BlipForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------
# Caption model
# -------------------------
blip_processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to(device)

# -------------------------
# CLIP embeddings
# -------------------------
clip_model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
).to(device)

clip_processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)

# -------------------------
# ControlNet
# -------------------------
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-scribble",
    torch_dtype=torch.float16
)

pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16
).to(device)

pipe.enable_attention_slicing()

# -------------------------
# Feedback dataset
# -------------------------
embeddings = []
labels = []

# -------------------------
# Caption from sketch
# -------------------------
def caption_image(image):

    inputs = blip_processor(
        image,
        return_tensors="pt"
    ).to(device)

    out = blip_model.generate(**inputs)

    caption = blip_processor.decode(
        out[0],
        skip_special_tokens=True
    )

    # remove words that bias art styles
    caption = caption.replace("drawing of", "")
    caption = caption.replace("sketch of", "")
    caption = caption.replace("a drawing", "")

    return caption.strip()


# -------------------------
# Prompt builder
# -------------------------
def build_prompt(caption, style):

    if style == "photorealistic":
        style_prompt = "ultra realistic photo, DSLR, natural lighting, sharp focus"

    elif style == "anime":
        style_prompt = "anime style, vibrant colors"

    elif style == "studio ghibli":
        style_prompt = "studio ghibli style, soft lighting, whimsical"

    elif style == "oil painting":
        style_prompt = "oil painting on canvas, thick brush strokes"

    elif style == "pencil sketch":
        style_prompt = "detailed pencil sketch"

    else:
        style_prompt = style

    return f"{caption}, {style_prompt}, highly detailed"


# -------------------------
# Generate image
# -------------------------
def generate(image, style):

    if image is None:
        return None, "No image"

    if isinstance(image, dict):
        image = image.get("composite", None)

    img_np = np.array(image)

    caption = caption_image(image)

    # Edge detection
    edge = cv2.Canny(img_np, 100, 200)
    edge = np.stack([edge]*3, axis=2)
    edge = Image.fromarray(edge).resize((512,512))

    prompt = build_prompt(caption, style)

    negative_prompt = "painting, oil painting, illustration, cartoon, blurry, low quality"

    result = pipe(
        prompt=prompt,
        image=edge,
        negative_prompt=negative_prompt,
        controlnet_conditioning_scale=1.5
    ).images[0]

    return result, caption


# -------------------------
# Save feedback
# -------------------------
def save_feedback(image, correct_label):

    try:

        if image is None:
            return "No image"

        if isinstance(image, dict):
            image = image.get("composite", None)

        inputs = clip_processor(
            images=image,
            return_tensors="pt"
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            emb = clip_model.get_image_features(**inputs)

        embeddings.append(emb.cpu().numpy())
        labels.append(correct_label)

        return f"Saved examples: {len(labels)}"

    except Exception as e:
        return f"Error: {str(e)}"


# -------------------------
# UI
# -------------------------
with gr.Blocks() as demo:

    gr.Markdown("# Sketch → AI Art")

    canvas = gr.ImageEditor(type="pil")

    style = gr.Dropdown(
        [
            "photorealistic",
            "anime",
            "studio ghibli",
            "oil painting",
            "pencil sketch"
        ],
        value="photorealistic"
    )

    btn = gr.Button("Generate")

    output = gr.Image()
    caption_box = gr.Textbox(label="AI Description")

    correct_label = gr.Textbox(label="Correction")
    feedback_btn = gr.Button("Save Feedback")
    status = gr.Textbox()

    btn.click(
        generate,
        inputs=[canvas, style],
        outputs=[output, caption_box]
    )

    feedback_btn.click(
        save_feedback,
        inputs=[canvas, correct_label],
        outputs=status
    )

demo.launch()

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://99ea719d1ef2e1650e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [7]:
import torch
import cv2
import numpy as np
import gradio as gr
from PIL import Image

from diffusers import StableDiffusionControlNetPipeline, ControlNetModel
from transformers import CLIPProcessor, CLIPModel
from transformers import BlipProcessor, BlipForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------
# Caption model
# -------------------------
blip_processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to(device)

# -------------------------
# CLIP embeddings
# -------------------------
clip_model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
).to(device)

clip_processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)

# -------------------------
# ControlNet
# -------------------------
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-scribble",
    torch_dtype=torch.float16
)

pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16
).to(device)

pipe.enable_attention_slicing()

# -------------------------
# Feedback dataset
# -------------------------
embeddings = []
labels = []

# -------------------------
# Caption from sketch
# -------------------------
def caption_image(image):

    inputs = blip_processor(
        image,
        return_tensors="pt"
    ).to(device)

    out = blip_model.generate(**inputs)

    caption = blip_processor.decode(
        out[0],
        skip_special_tokens=True
    )

    # remove words that bias art styles
    caption = caption.replace("drawing of", "")
    caption = caption.replace("sketch of", "")
    caption = caption.replace("a drawing", "")

    return caption.strip()


# -------------------------
# Prompt builder
# -------------------------
def build_prompt(caption, style):

    if style == "photorealistic":
        style_prompt = "ultra realistic photo, DSLR, natural lighting, sharp focus"

    elif style == "anime":
        style_prompt = "anime style, vibrant colors"

    elif style == "studio ghibli":
        style_prompt = "studio ghibli style, soft lighting, whimsical"

    elif style == "oil painting":
        style_prompt = "oil painting on canvas, thick brush strokes"

    elif style == "pencil sketch":
        style_prompt = "detailed pencil sketch"

    else:
        style_prompt = style

    return f"{caption}, {style_prompt}, highly detailed"


# -------------------------
# Generate image
# -------------------------
def generate(image, style):

    if image is None:
        return None, "No image"

    if isinstance(image, dict):
        image = image.get("composite", None)

    img_np = np.array(image)

    caption = caption_image(image)

    # Edge detection
    edge = cv2.Canny(img_np, 100, 200)
    edge = np.stack([edge]*3, axis=2)
    edge = Image.fromarray(edge).resize((512,512))

    prompt = build_prompt(caption, style)

    negative_prompt = "painting, oil painting, illustration, cartoon, blurry, low quality"

    result = pipe(
        prompt=prompt,
        image=edge,
        negative_prompt=negative_prompt,
        controlnet_conditioning_scale=1.5
    ).images[0]

    return result, caption


# -------------------------
# Save feedback
# -------------------------
def save_feedback(image, correct_label):

    try:

        if image is None:
            return "No image"

        if isinstance(image, dict):
            image = image.get("composite", None)

        inputs = clip_processor(
            images=image,
            return_tensors="pt"
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            emb = clip_model.get_image_features(**inputs)

        embeddings.append(emb.cpu().numpy())
        labels.append(correct_label)

        return f"Saved examples: {len(labels)}"

    except Exception as e:
        return f"Error: {str(e)}"


# -------------------------
# UI
# -------------------------
with gr.Blocks() as demo:

    gr.Markdown("# Sketch → AI Art")

    canvas = gr.ImageEditor(type="pil")

    style = gr.Dropdown(
        [
            "photorealistic",
            "anime",
            "studio ghibli",
            "oil painting",
            "pencil sketch"
        ],
        value="photorealistic"
    )

    btn = gr.Button("Generate")

    output = gr.Image()
    caption_box = gr.Textbox(label="AI Description")

    correct_label = gr.Textbox(label="Correction")
    feedback_btn = gr.Button("Save Feedback")
    status = gr.Textbox()

    btn.click(
        generate,
        inputs=[canvas, style],
        outputs=[output, caption_box]
    )

    feedback_btn.click(
        save_feedback,
        inputs=[canvas, correct_label],
        outputs=status
    )

demo.launch()

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d65f010a92245b145c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [8]:
import torch
import cv2
import numpy as np
import gradio as gr
from PIL import Image

from diffusers import StableDiffusionControlNetPipeline, ControlNetModel
from transformers import CLIPProcessor, CLIPModel
from transformers import BlipProcessor, BlipForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------
# Caption model
# -------------------------
blip_processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to(device)

# -------------------------
# CLIP embeddings
# -------------------------
clip_model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
).to(device)

clip_processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)

# -------------------------
# ControlNet
# -------------------------
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-scribble",
    torch_dtype=torch.float16
)

pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16
).to(device)

pipe.enable_attention_slicing()

# -------------------------
# Feedback dataset
# -------------------------
embeddings = []
labels = []

# -------------------------
# Caption from sketch
# -------------------------
def caption_image(image):

    inputs = blip_processor(
        image,
        return_tensors="pt"
    ).to(device)

    out = blip_model.generate(**inputs)

    caption = blip_processor.decode(
        out[0],
        skip_special_tokens=True
    )

    # remove words that bias art styles
    caption = caption.replace("drawing of", "")
    caption = caption.replace("sketch of", "")
    caption = caption.replace("a drawing", "")

    return caption.strip()


# -------------------------
# Prompt builder
# -------------------------
def build_prompt(caption, style):

    if style == "photorealistic":
        style_prompt = "ultra realistic photo, DSLR, natural lighting, sharp focus"

    elif style == "anime":
        style_prompt = "anime style, vibrant colors"

    elif style == "studio ghibli":
        style_prompt = "studio ghibli style, soft lighting, whimsical"

    elif style == "oil painting":
        style_prompt = "oil painting on canvas, thick brush strokes"

    elif style == "pencil sketch":
        style_prompt = "detailed pencil sketch"

    else:
        style_prompt = style

    return f"{caption}, {style_prompt}, highly detailed"


# -------------------------
# Generate image
# -------------------------
def generate(image, style):

    if image is None:
        return None, "No image"

    if isinstance(image, dict):
        image = image.get("composite", None)

    img_np = np.array(image)

    caption = caption_image(image)

    # Edge detection
    edge = cv2.Canny(img_np, 100, 200)
    edge = np.stack([edge]*3, axis=2)
    edge = Image.fromarray(edge).resize((512,512))

    prompt = build_prompt(caption, style)

    negative_prompt = "painting, oil painting, illustration, cartoon, blurry, low quality"

    result = pipe(
        prompt=prompt,
        image=edge,
        negative_prompt=negative_prompt,
        controlnet_conditioning_scale=1.5
    ).images[0]

    return result, caption


# -------------------------
# Save feedback
# -------------------------
def save_feedback(image, correct_label):

    try:

        if image is None:
            return "No image"

        if isinstance(image, dict):
            image = image.get("composite", None)

        inputs = clip_processor(
            images=image,
            return_tensors="pt"
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            emb = clip_model.get_image_features(**inputs)

        embeddings.append(emb.cpu().numpy())
        labels.append(correct_label)

        return f"Saved examples: {len(labels)}"

    except Exception as e:
        return f"Error: {str(e)}"


# -------------------------
# UI
# -------------------------
with gr.Blocks() as demo:

    gr.Markdown("# Sketch → AI Art")

    canvas = gr.ImageEditor(type="pil")

    style = gr.Dropdown(
        [
            "photorealistic",
            "anime",
            "studio ghibli",
            "oil painting",
            "pencil sketch"
        ],
        value="photorealistic"
    )

    btn = gr.Button("Generate")

    output = gr.Image()
    caption_box = gr.Textbox(label="AI Description")

    correct_label = gr.Textbox(label="Correction")
    feedback_btn = gr.Button("Save Feedback")
    status = gr.Textbox()

    btn.click(
        generate,
        inputs=[canvas, style],
        outputs=[output, caption_box]
    )

    feedback_btn.click(
        save_feedback,
        inputs=[canvas, correct_label],
        outputs=status
    )

demo.launch()

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://59c57abda7d637c113.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [9]:
import torch
import cv2
import numpy as np
import gradio as gr
from PIL import Image

from diffusers import StableDiffusionControlNetPipeline, ControlNetModel
from transformers import CLIPProcessor, CLIPModel
from transformers import BlipProcessor, BlipForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------
# Caption model
# -------------------------
blip_processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to(device)

# -------------------------
# CLIP embeddings
# -------------------------
clip_model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
).to(device)

clip_processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)

# -------------------------
# ControlNet
# -------------------------
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-scribble",
    torch_dtype=torch.float16
)

pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16
).to(device)

pipe.enable_attention_slicing()

# -------------------------
# Feedback dataset
# -------------------------
embeddings = []
labels = []

# -------------------------
# Caption from sketch
# -------------------------
def caption_image(image):

    inputs = blip_processor(
        image,
        return_tensors="pt"
    ).to(device)

    out = blip_model.generate(**inputs)

    caption = blip_processor.decode(
        out[0],
        skip_special_tokens=True
    )

    # remove words that bias art styles
    caption = caption.replace("drawing of", "")
    caption = caption.replace("sketch of", "")
    caption = caption.replace("a drawing", "")

    return caption.strip()


# -------------------------
# Prompt builder
# -------------------------
def build_prompt(caption, style):

    if style == "photorealistic":
        style_prompt = "ultra realistic photo, DSLR, natural lighting, sharp focus"

    elif style == "anime":
        style_prompt = "anime style, vibrant colors"

    elif style == "studio ghibli":
        style_prompt = "studio ghibli style, soft lighting, whimsical"

    elif style == "oil painting":
        style_prompt = "oil painting on canvas, thick brush strokes"

    elif style == "pencil sketch":
        style_prompt = "detailed pencil sketch"

    else:
        style_prompt = style

    return f"{caption}, {style_prompt}, highly detailed"


# -------------------------
# Generate image
# -------------------------
def generate(image, style):

    if image is None:
        return None, "No image"

    if isinstance(image, dict):
        image = image.get("composite", None)

    img_np = np.array(image)

    caption = caption_image(image)

    # Edge detection
    edge = cv2.Canny(img_np, 100, 200)
    edge = np.stack([edge]*3, axis=2)
    edge = Image.fromarray(edge).resize((512,512))

    prompt = build_prompt(caption, style)

    negative_prompt = "painting, oil painting, illustration, cartoon, blurry, low quality"

    result = pipe(
        prompt=prompt,
        image=edge,
        negative_prompt=negative_prompt,
        controlnet_conditioning_scale=1.5
    ).images[0]

    return result, caption


# -------------------------
# Save feedback
# -------------------------
def save_feedback(image, correct_label):

    try:

        if image is None:
            return "No image"

        if isinstance(image, dict):
            image = image.get("composite", None)

        inputs = clip_processor(
            images=image,
            return_tensors="pt"
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            emb = clip_model.get_image_features(**inputs)

        embeddings.append(emb.cpu().numpy())
        labels.append(correct_label)

        return f"Saved examples: {len(labels)}"

    except Exception as e:
        return f"Error: {str(e)}"


# -------------------------
# UI
# -------------------------
with gr.Blocks() as demo:

    gr.Markdown("# Sketch → AI Art")

    canvas = gr.ImageEditor(type="pil")

    style = gr.Dropdown(
        [
            "photorealistic",
            "anime",
            "studio ghibli",
            "oil painting",
            "pencil sketch"
        ],
        value="photorealistic"
    )

    btn = gr.Button("Generate")

    output = gr.Image()
    caption_box = gr.Textbox(label="AI Description")

    correct_label = gr.Textbox(label="Correction")
    feedback_btn = gr.Button("Save Feedback")
    status = gr.Textbox()

    btn.click(
        generate,
        inputs=[canvas, style],
        outputs=[output, caption_box]
    )

    feedback_btn.click(
        save_feedback,
        inputs=[canvas, correct_label],
        outputs=status
    )

demo.launch()

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://912fb93b04b24eaf51.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
